# Axis — COMPLETE 27-class ECG model (Colab)

Trains the full rhythm+diagnostic taxonomy (AF, flutter, brady/tachy, ectopy,
conduction, axis, intervals, ST/T, hypertrophy, MI) on PTB-XL + CinC2020 from
the CODE-15 backbone. Produces `checkpoint.pt` with per-class AUROC.

**Set `Runtime -> Change runtime type -> GPU` first.** Then run cells in order.
You'll upload your `backbone.pt` (from the CODE-15 pretrain) in cell 4.

In [ ]:
#@title 1) GPU + code
!nvidia-smi --query-gpu=name --format=csv,noheader || echo 'set GPU runtime'
%cd /content
!rm -rf Heartcheck && git clone -q https://github.com/Davmunrey/Heartcheck
%cd /content/Heartcheck

In [ ]:
#@title 2) Deps (aria2 + torch/numpy/h5py/pyarrow/wfdb + heartscan_ml)
!command -v aria2c >/dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y aria2)
!pip install -q numpy h5py pyarrow wfdb scipy
!pip install -q --no-deps -e ml/
import torch; print('CUDA:', torch.cuda.is_available())

In [ ]:
#@title 3) Download PTB-XL + CinC2020 from PhysioNet (~10 GB, fast)
import os; os.makedirs('/content/data', exist_ok=True)
%cd /content/data
PTB='https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip'
CINC='https://physionet.org/static/published-projects/challenge-2020/classification-of-12-lead-ecgs-the-physionetcomputing-in-cardiology-challenge-2020-1.0.2.zip'
!aria2c -c -x16 -s16 --console-log-level=warn -o ptbxl.zip "$PTB" && unzip -q -o ptbxl.zip && rm ptbxl.zip
!aria2c -c -x16 -s16 --console-log-level=warn -o cinc.zip "$CINC" && unzip -q -o cinc.zip && rm cinc.zip
# symlink to the names the manifest CLI expects
!mkdir -p ~/heartscan_data
!ln -sfn /content/data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3 ~/heartscan_data/ptb_xl
!ln -sfn /content/data/classification-of-12-lead-ecgs-the-physionetcomputing-in-cardiology-challenge-2020-1.0.2 ~/heartscan_data/cinc2020
!ls ~/heartscan_data/ptb_xl/records100 | head -2; ls ~/heartscan_data/cinc2020/training | head
%cd /content/Heartcheck

In [ ]:
#@title 4) Upload your backbone.pt (from the CODE-15 pretrain, ~27 MB)
import os; os.makedirs('runs/local/code15_pretrain', exist_ok=True)
from google.colab import files
up = files.upload()   # pick backbone.pt
import shutil; [shutil.move(k, 'runs/local/code15_pretrain/backbone.pt') for k in up]
print('backbone ready:', os.path.exists('runs/local/code15_pretrain/backbone.pt'))

In [ ]:
#@title 5) Build the 27-class manifest (PTB-XL + CinC2020)
!python -m ml.datasets.cli manifest --root ~/heartscan_data --datasets ptb_xl cinc2020 --out runs/local/blend27/manifest.parquet
!python -m ml.datasets.splits --manifest runs/local/blend27/manifest.parquet --out runs/local/blend27/manifest_split.parquet

In [ ]:
#@title 6) Train the 27-class complete model (CUDA, ~1-2h)
EPOCHS=20  #@param {type:"integer"}
BATCH=128  #@param {type:"integer"}
!python -m ml.training.train_multilabel27 \
  --manifest runs/local/blend27/manifest_split.parquet --out runs/local/full27 \
  --init-backbone runs/local/code15_pretrain/backbone.pt \
  --epochs $EPOCHS --batch-size $BATCH --lr 3e-4 --workers 8

In [ ]:
#@title 7) Download the trained 27-class checkpoint + see per-class AUROC
import json; print(json.load(open('runs/local/full27/summary.json')))
from google.colab import files; files.download('runs/local/full27/checkpoint.pt')

Trae `checkpoint.pt` al portatil; tiene `classes` (las 27) + `per_class_auroc`.
Despues lo servimos en la API como el modelo completo (panel de 27 afecciones).